# Refine training masks (pecan / kernel / crack)

**Section 1** — separate mask TIFFs listed in `files to use for training.txt`.

**Section 2** — combined multi-class TIFFs under `D:\VIDS & MASKS` (`* - Pecan,Kernel,Crack.tiff`).

For both formats:

1. Load pecan, kernel, and crack masks (or extract them from one combined label image).
2. Walk frames in lockstep.
3. Clip kernel and crack to the pecan region (drop pixels outside pecan).
4. Where kernel and crack overlap, keep **kernel** only and remove that region from crack.
5. Save in place (pecan pixels / files are left unchanged).

Run cells top to bottom.

In [1]:
from __future__ import annotations

import re
from collections import defaultdict
from pathlib import Path

import numpy as np
from scipy import ndimage
from tqdm.auto import tqdm

from napari_pecan_py.widgets.yolo_seg.model import load_mask_volume, save_mask_volume

REPO_ROOT = Path.cwd()
FILE_LIST = REPO_ROOT / "files to use for training.txt"

# Set False first if you want a dry run (no writes).
SAVE_IN_PLACE = True

In [4]:
CLASS_SUFFIX_RE = re.compile(r" - (Pecan|Kernel|Crack)$", re.IGNORECASE)


def parse_mask_path(path: Path) -> tuple[str, str] | None:
    """Return (video_key, class_name) for a mask file path."""
    match = CLASS_SUFFIX_RE.search(path.stem)
    if match is None:
        return None
    class_name = match.group(1).capitalize()  # Pecan, Kernel, Crack
    video_key = str(path.parent / path.stem[: match.start()])
    return video_key, class_name


def group_mask_files(paths: list[Path]) -> dict[str, dict[str, Path]]:
    groups: dict[str, dict[str, Path]] = defaultdict(dict)
    skipped: list[Path] = []
    for path in paths:
        parsed = parse_mask_path(path)
        if parsed is None:
            skipped.append(path)
            continue
        video_key, class_name = parsed
        groups[video_key][class_name] = path
    return dict(groups), skipped


def as_frame_stack(volume: np.ndarray) -> np.ndarray:
    arr = np.asarray(volume)
    if arr.ndim == 2:
        return arr[np.newaxis, ...]
    if arr.ndim == 3:
        return arr
    raise ValueError(f"Expected 2D or 3D mask volume, got shape {arr.shape}")


def refine_frame(
    pecan: np.ndarray,
    kernel: np.ndarray | None,
    crack: np.ndarray | None,
) -> tuple[np.ndarray | None, np.ndarray | None]:
    """Clip kernel/crack to pecan and resolve kernel/crack overlap."""
    pecan_fg = pecan > 0

    if kernel is not None:
        kernel = np.where(pecan_fg, kernel, 0).astype(kernel.dtype, copy=False)
    if crack is not None:
        crack = np.where(pecan_fg, crack, 0).astype(crack.dtype, copy=False)
    if kernel is not None and crack is not None:
        crack = np.where(kernel > 0, 0, crack).astype(crack.dtype, copy=False)

    return kernel, crack


def refine_video_masks(
    pecan_path: Path,
    kernel_path: Path | None,
    crack_path: Path | None,
    *,
    save: bool = True,
) -> dict[str, int]:
    pecan_raw = load_mask_volume(pecan_path)
    pecan_vol = as_frame_stack(pecan_raw)

    kernel_raw = load_mask_volume(kernel_path) if kernel_path else None
    crack_raw = load_mask_volume(crack_path) if crack_path else None
    kernel_vol = as_frame_stack(kernel_raw) if kernel_raw is not None else None
    crack_vol = as_frame_stack(crack_raw) if crack_raw is not None else None

    n_frames = pecan_vol.shape[0]
    if kernel_vol is not None and kernel_vol.shape[0] != n_frames:
        raise ValueError(
            f"Frame count mismatch: pecan={n_frames}, kernel={kernel_vol.shape[0]} ({kernel_path})"
        )
    if crack_vol is not None and crack_vol.shape[0] != n_frames:
        raise ValueError(
            f"Frame count mismatch: pecan={n_frames}, crack={crack_vol.shape[0]} ({crack_path})"
        )

    stats = {
        "frames": n_frames,
        "kernel_clipped_px": 0,
        "crack_clipped_px": 0,
        "crack_removed_for_kernel_px": 0,
    }

    out_kernel_frames: list[np.ndarray] = []
    out_crack_frames: list[np.ndarray] = []

    frame_iter = range(n_frames)
    for t in frame_iter:
        pecan = pecan_vol[t]
        kernel = kernel_vol[t] if kernel_vol is not None else None
        crack = crack_vol[t] if crack_vol is not None else None

        if kernel is not None:
            stats["kernel_clipped_px"] += int(np.count_nonzero((kernel > 0) & ~(pecan > 0)))
        if crack is not None:
            stats["crack_clipped_px"] += int(np.count_nonzero((crack > 0) & ~(pecan > 0)))
            if kernel is not None:
                stats["crack_removed_for_kernel_px"] += int(
                    np.count_nonzero((crack > 0) & (kernel > 0))
                )

        kernel, crack = refine_frame(pecan, kernel, crack)

        if kernel is not None:
            out_kernel_frames.append(kernel)
        if crack is not None:
            out_crack_frames.append(crack)

    if save:
        if kernel_path is not None and out_kernel_frames:
            kernel_out = (
                out_kernel_frames[0]
                if np.ndim(kernel_raw) == 2
                else np.stack(out_kernel_frames, axis=0)
            )
            save_mask_volume(kernel_out, kernel_path, "tiff")
        if crack_path is not None and out_crack_frames:
            crack_out = (
                out_crack_frames[0]
                if np.ndim(crack_raw) == 2
                else np.stack(out_crack_frames, axis=0)
            )
            save_mask_volume(crack_out, crack_path, "tiff")

    return stats


# --- Combined single-TIFF labels (3=Pecan, 2=Kernel, 1=Crack) ---

COMBINED_LABEL_PECAN = 3
COMBINED_LABEL_KERNEL = 2
COMBINED_LABEL_CRACK = 1
COMBINED_MASK_STEM_SUFFIX = " - Pecan,Kernel,Crack"


def discover_combined_mask_files(root: Path) -> list[Path]:
    """Find multi-class label TIFFs under ``root``."""
    paths: list[Path] = []
    for pattern in ("*.tiff", "*.tif"):
        for path in root.rglob(pattern):
            if path.stem.endswith(COMBINED_MASK_STEM_SUFFIX):
                paths.append(path)
    return sorted(paths, key=lambda p: str(p).lower())


def combined_pecan_clip_mask(labels: np.ndarray) -> np.ndarray:
    """Foreground connected to pecan (label 3), dropping stray disconnected blobs."""
    labels = np.asarray(labels)
    fg = labels > 0
    if not np.any(fg):
        return np.zeros_like(labels, dtype=labels.dtype)

    labeled, _ = ndimage.label(fg)
    pecan_seed = labels == COMBINED_LABEL_PECAN
    if not np.any(pecan_seed):
        counts = np.bincount(labeled.ravel())
        counts[0] = 0
        largest = int(counts.argmax())
        return (labeled == largest).astype(labels.dtype)

    keep_ids = np.unique(labeled[pecan_seed])
    keep_ids = keep_ids[keep_ids != 0]
    return np.isin(labeled, keep_ids).astype(labels.dtype)


def refine_combined_frame(labels: np.ndarray) -> tuple[np.ndarray, dict[str, int]]:
    """Refine one frame of a combined pecan/kernel/crack label image."""
    labels = np.asarray(labels)
    pecan_unchanged = labels == COMBINED_LABEL_PECAN
    pecan_clip = combined_pecan_clip_mask(labels)

    kernel = np.where(labels == COMBINED_LABEL_KERNEL, COMBINED_LABEL_KERNEL, 0)
    crack = np.where(labels == COMBINED_LABEL_CRACK, COMBINED_LABEL_CRACK, 0)

    stats = {
        "kernel_clipped_px": int(np.count_nonzero((kernel > 0) & ~(pecan_clip > 0))),
        "crack_clipped_px": int(np.count_nonzero((crack > 0) & ~(pecan_clip > 0))),
        "crack_removed_for_kernel_px": int(np.count_nonzero((crack > 0) & (kernel > 0))),
    }

    kernel, crack = refine_frame(pecan_clip, kernel, crack)

    out = np.zeros_like(labels)
    out[pecan_unchanged] = COMBINED_LABEL_PECAN
    out[kernel > 0] = COMBINED_LABEL_KERNEL
    out[crack > 0] = COMBINED_LABEL_CRACK
    return out, stats


def refine_combined_mask(path: Path, *, save: bool = True) -> dict[str, int]:
    """Load, refine, and optionally overwrite a combined multi-class mask TIFF."""
    raw = load_mask_volume(path)
    volume = as_frame_stack(raw)

    stats = {
        "frames": volume.shape[0],
        "kernel_clipped_px": 0,
        "crack_clipped_px": 0,
        "crack_removed_for_kernel_px": 0,
    }
    out_frames: list[np.ndarray] = []

    for frame in volume:
        refined, frame_stats = refine_combined_frame(frame)
        for key in ("kernel_clipped_px", "crack_clipped_px", "crack_removed_for_kernel_px"):
            stats[key] += frame_stats[key]
        out_frames.append(refined)

    if save:
        out = out_frames[0] if np.ndim(raw) == 2 else np.stack(out_frames, axis=0)
        save_mask_volume(out, path, "tiff")

    return stats

In [3]:
raw_lines = [line.strip() for line in FILE_LIST.read_text(encoding="utf-8").splitlines() if line.strip()]
all_paths = [Path(p) for p in raw_lines]
groups, skipped_paths = group_mask_files(all_paths)

jobs: list[tuple[str, Path, Path | None, Path | None]] = []
pecan_only = 0

for video_key, class_paths in sorted(groups.items()):
    pecan_path = class_paths.get("Pecan")
    kernel_path = class_paths.get("Kernel")
    crack_path = class_paths.get("Crack")

    if pecan_path is None:
        continue
    if kernel_path is None and crack_path is None:
        pecan_only += 1
        continue

    jobs.append((video_key, pecan_path, kernel_path, crack_path))

print(f"Listed mask files: {len(all_paths)}")
print(f"Unique videos: {len(groups)}")
print(f"Videos to refine (pecan + kernel and/or crack): {len(jobs)}")
print(f"Pecan-only videos (skipped): {pecan_only}")
if skipped_paths:
    print(f"Unrecognized paths (skipped): {len(skipped_paths)}")
    for p in skipped_paths[:5]:
        print(f"  - {p}")
    if len(skipped_paths) > 5:
        print(f"  ... and {len(skipped_paths) - 5} more")

Listed mask files: 174
Unique videos: 102
Videos to refine (pecan + kernel and/or crack): 60
Pecan-only videos (skipped): 42


In [4]:
errors: list[tuple[str, str]] = []
totals = {
    "frames": 0,
    "kernel_clipped_px": 0,
    "crack_clipped_px": 0,
    "crack_removed_for_kernel_px": 0,
}

for video_key, pecan_path, kernel_path, crack_path in tqdm(jobs, desc="Videos", unit="video"):
    label = Path(video_key).name
    classes = ["pecan"]
    if kernel_path:
        classes.append("kernel")
    if crack_path:
        classes.append("crack")

    try:
        stats = refine_video_masks(
            pecan_path,
            kernel_path,
            crack_path,
            save=SAVE_IN_PLACE,
        )
        for k in totals:
            totals[k] += stats[k]
        tqdm.write(
            f"{label}: {stats['frames']} frames | "
            f"kernel clipped {stats['kernel_clipped_px']} px | "
            f"crack clipped {stats['crack_clipped_px']} px | "
            f"crack→kernel overlap removed {stats['crack_removed_for_kernel_px']} px"
        )
    except Exception as exc:
        errors.append((label, str(exc)))
        tqdm.write(f"ERROR {label}: {exc}")

print("\n--- Done ---")
print(f"SAVE_IN_PLACE = {SAVE_IN_PLACE}")
print(f"Processed videos: {len(jobs) - len(errors)} / {len(jobs)}")
print(f"Total frames: {totals['frames']}")
print(f"Kernel pixels clipped outside pecan: {totals['kernel_clipped_px']}")
print(f"Crack pixels clipped outside pecan: {totals['crack_clipped_px']}")
print(f"Crack pixels reassigned to kernel overlap: {totals['crack_removed_for_kernel_px']}")
if errors:
    print(f"\nErrors ({len(errors)}):")
    for label, msg in errors:
        print(f"  - {label}: {msg}")

Videos:   0%|          | 0/60 [00:00<?, ?video/s]

GH013515-cropped: 272 frames | kernel clipped 936 px | crack clipped 0 px | crack→kernel overlap removed 1329193 px
GH013516-cropped: 268 frames | kernel clipped 8399 px | crack clipped 0 px | crack→kernel overlap removed 872032 px
GH013517-cropped: 236 frames | kernel clipped 0 px | crack clipped 0 px | crack→kernel overlap removed 0 px
GH013518-cropped: 135 frames | kernel clipped 19574 px | crack clipped 0 px | crack→kernel overlap removed 0 px
GH013519-cropped: 266 frames | kernel clipped 12461 px | crack clipped 0 px | crack→kernel overlap removed 962376 px
GH013520-cropped: 245 frames | kernel clipped 8499 px | crack clipped 0 px | crack→kernel overlap removed 1686580 px
GH013550-cropped: 233 frames | kernel clipped 0 px | crack clipped 0 px | crack→kernel overlap removed 0 px
GH013551-cropped: 219 frames | kernel clipped 0 px | crack clipped 0 px | crack→kernel overlap removed 0 px
GH013552-cropped: 213 frames | kernel clipped 0 px | crack clipped 0 px | crack→kernel overlap rem

## Combined labels under `D:\VIDS & MASKS`

Some videos have a **single** TIFF next to the video, named like `GH013810-cropped - Pecan,Kernel,Crack.tiff`, with class IDs:

| Value | Class |
|------:|-------|
| 3 | Pecan |
| 2 | Kernel |
| 1 | Crack |

The same rules apply: pecan pixels (3) are untouched; kernel and crack are clipped to foreground connected to pecan and kernel wins on overlap. Stray kernel/crack blobs disconnected from the pecan region are removed. Files are overwritten in place.

In [6]:
COMBINED_ROOT = Path(r"D:\VIDS & MASKS")

combined_jobs = discover_combined_mask_files(COMBINED_ROOT)
print(f"Scanning: {COMBINED_ROOT}")
print(f"Combined mask TIFFs found: {len(combined_jobs)}")
if combined_jobs:
    print(f"Example: {combined_jobs[0].name}")

Scanning: D:\VIDS & MASKS
Combined mask TIFFs found: 1511
Example: GH013863-cropped - Pecan,Kernel,Crack.tiff


In [ ]:
combined_errors: list[tuple[str, str]] = []
combined_totals = {
    "frames": 0,
    "kernel_clipped_px": 0,
    "crack_clipped_px": 0,
    "crack_removed_for_kernel_px": 0,
}

for mask_path in tqdm(combined_jobs, desc="Combined masks", unit="video"):
    label = mask_path.stem
    try:
        stats = refine_combined_mask(mask_path, save=SAVE_IN_PLACE)
        for key in combined_totals:
            combined_totals[key] += stats[key]
        tqdm.write(
            f"{label}: {stats['frames']} frames | "
            f"kernel clipped {stats['kernel_clipped_px']} px | "
            f"crack clipped {stats['crack_clipped_px']} px | "
            f"crack→kernel overlap removed {stats['crack_removed_for_kernel_px']} px"
        )
    except Exception as exc:
        combined_errors.append((label, str(exc)))
        tqdm.write(f"ERROR {label}: {exc}")

print("\n--- Combined labels done ---")
print(f"SAVE_IN_PLACE = {SAVE_IN_PLACE}")
print(f"Processed: {len(combined_jobs) - len(combined_errors)} / {len(combined_jobs)}")
print(f"Total frames: {combined_totals['frames']}")
print(f"Kernel pixels clipped outside pecan: {combined_totals['kernel_clipped_px']}")
print(f"Crack pixels clipped outside pecan: {combined_totals['crack_clipped_px']}")
print(f"Crack pixels removed for kernel overlap: {combined_totals['crack_removed_for_kernel_px']}")
if combined_errors:
    print(f"\nErrors ({len(combined_errors)}):")
    for label, msg in combined_errors:
        print(f"  - {label}: {msg}")

Combined masks:   0%|          | 0/1511 [00:00<?, ?video/s]

GH013863-cropped - Pecan,Kernel,Crack: 238 frames | kernel clipped 231 px | crack clipped 231 px | crack→kernel overlap removed 0 px
GH013864-cropped - Pecan,Kernel,Crack: 293 frames | kernel clipped 8 px | crack clipped 892 px | crack→kernel overlap removed 0 px
GH013865-cropped - Pecan,Kernel,Crack: 233 frames | kernel clipped 66 px | crack clipped 16537 px | crack→kernel overlap removed 0 px
GH013866-cropped - Pecan,Kernel,Crack: 249 frames | kernel clipped 0 px | crack clipped 6305 px | crack→kernel overlap removed 0 px
GH013867-cropped - Pecan,Kernel,Crack: 251 frames | kernel clipped 0 px | crack clipped 791 px | crack→kernel overlap removed 0 px
GH013868-cropped - Pecan,Kernel,Crack: 249 frames | kernel clipped 11 px | crack clipped 30337 px | crack→kernel overlap removed 0 px
GH013869-cropped - Pecan,Kernel,Crack: 236 frames | kernel clipped 0 px | crack clipped 13947 px | crack→kernel overlap removed 0 px
GH013870-cropped - Pecan,Kernel,Crack: 177 frames | kernel clipped 0 px 